# Movie Rating Prediction with Python and Machine Learning

This notebook contains the complete analysis and modeling workflow to predict IMDb movie ratings for Indian cinema based on historical movie metadata.

## Workflow Outline:
1. Import libraries
2. Load dataset
3. Inspect data structure & missingness
4. Preprocess features and targets
5. Exploratory Data Analysis (EDA)
6. Feature Engineering & ColumnTransformer pipeline
7. Train/Test split
8. Model training, comparison and Cross-Validation
9. Hyperparameter tuning
10. Final model evaluation and interpretability
11. Saving the model pipeline

### 1. Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

import sys
sys.path.append(os.path.abspath('../'))
from src.data_preprocessing import clean_target_and_duplicates, MovieDataCleaner, GenreBinarizer
from src.feature_engineering import build_preprocessor

sns.set_theme(style="whitegrid")
print("Libraries loaded successfully.")

### 2. Load Dataset

In [ ]:
data_path = '../data/IMDb Movies India.csv'
df = pd.read_csv(data_path, encoding='latin1')
print(f"Dataset Shape: {df.shape}")
df.head()

### 3. Inspect Data

In [ ]:
print("--- Missing Values Count ---")
print(df.isnull().sum())
print("\n--- Data Types ---")
print(df.dtypes)

### 4. Clean Data
We drop records missing the target rating and remove duplicates. We then separate features and targets.

In [ ]:
df_clean = clean_target_and_duplicates(df)
print(f"Cleaned dataset size: {df_clean.shape}")
X = df_clean.drop(columns=['Rating'])
y = df_clean['Rating']

### 5. Exploratory Data Analysis (EDA)
Let's clean Year, Duration, and Votes temporarily to visualize relationships.

In [ ]:
cleaner = MovieDataCleaner()
df_eda = cleaner.fit_transform(df_clean)

# 1. Rating Distribution
plt.figure(figsize=(8, 5))
sns.histplot(df_eda['Rating'], kde=True, color='teal')
plt.title('Distribution of IMDb Ratings')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.show()

# 2. Rating vs Votes
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Votes', y='Rating', data=df_eda, alpha=0.3, color='purple')
plt.xscale('log')
plt.title('IMDb Rating vs Votes (Log Scale)')
plt.xlabel('Number of Votes')
plt.ylabel('Rating')
plt.show()

# 3. Rating vs Duration
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Duration', y='Rating', data=df_eda, alpha=0.3, color='darkorange')
plt.title('IMDb Rating vs Duration')
plt.xlabel('Duration (min)')
plt.ylabel('Rating')
plt.show()

### 6. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

### 7. Model Training and Cross-Validation
Compare Linear Regression, Ridge, Random Forest, and Gradient Boosting Regressors.

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'Hist Gradient Boosting': HistGradientBoostingRegressor(random_state=42)
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
results = []

for name, model in models.items():
    preprocessor = build_preprocessor()
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    scores = cross_validate(
        pipeline, X_train, y_train, cv=cv,
        scoring={'mae': 'neg_mean_absolute_error', 'rmse': 'neg_root_mean_squared_error', 'r2': 'r2'},
        n_jobs=-1
    )
    
    results.append({
        'Model': name,
        'CV MAE': -scores['test_mae'].mean(),
        'CV RMSE': -scores['test_rmse'].mean(),
        'CV R2': scores['test_r2'].mean()
    })

results_df = pd.DataFrame(results)
results_df

### 8. Hyperparameter Tuning
Tune the best-performing model (Hist Gradient Boosting).

In [ ]:
param_grid = {
    'regressor__learning_rate': [0.05, 0.1, 0.2],
    'regressor__max_iter': [100, 150],
    'regressor__max_depth': [5, 10, None]
}

tune_preprocessor = build_preprocessor()
tune_pipeline = Pipeline(steps=[
    ('preprocessor', tune_preprocessor),
    ('regressor', HistGradientBoostingRegressor(random_state=42))
])

grid_search = GridSearchCV(
    tune_pipeline, param_grid, cv=cv,
    scoring='neg_root_mean_squared_error', n_jobs=-1
)
grid_search.fit(X_train, y_train)

best_pipeline = grid_search.best_estimator_
print(f"Best parameters: {grid_search.best_params_}")

### 9. Evaluate Best Model on Test Set

In [ ]:
y_pred = best_pipeline.predict(X_test)
print(f"Test MAE: {mean_absolute_error(y_test, y_pred):.2f}")
print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")
print(f"Test R2 Score: {r2_score(y_test, y_pred):.2f}")

### 10. Model Interpretability: Permutation Feature Importance

In [ ]:
result_importances = permutation_importance(
    best_pipeline, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1
)
importances_df = pd.DataFrame({
    'Feature': X_test.columns,
    'Importance': result_importances.importances_mean
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=importances_df, palette='magma')
plt.title('Permutation Feature Importance on Test Set')
plt.xlabel('Importance (drop in R2)')
plt.ylabel('Feature')
plt.show()
print("Feature importance shows model association, not causation.")

### 11. Save the Model Pipeline

In [ ]:
import joblib
model_dir = '../models/'
os.makedirs(model_dir, exist_ok=True)
joblib.dump(best_pipeline, os.path.join(model_dir, 'movie_rating_model.pkl'))
print("Model pipeline saved to models/movie_rating_model.pkl.")